# 125.785 Research and Analytics in Economics and Finance
## Session 10: Event studies: earnings announcements

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nquynh/125785-research-analytics/blob/main/notebooks/S10_student.ipynb)

**Reading:** Brooks (2019) Ch. 14.1–14.2  
**Data:** `vn_earnings_events.csv`

**By the end of this session you can:**
- Estimate normal returns with the market model
- Compute abnormal returns (AR), CAR and CAAR
- Test CAAR significance
- Explain CARs with a cross-sectional regression

*All datasets are simulated for teaching. They mimic the structure of Vietnamese listed firms, banks and markets, but are not real company data.*

### 0. Setup

In [ ]:
# Run this cell first in every session
!pip -q install linearmodels        # statsmodels, pandas, numpy, matplotlib are pre-installed on Colab
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf
pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 30)

# Where the course data live (GitHub). If you uploaded the CSV files to Colab, use DATA = ""
DATA = os.environ.get("DATA_DIR", "https://raw.githubusercontent.com/nquynh/125785-research-analytics/main/data/")

### 1. Worked example

`vn_earnings_events.csv` contains 400 quarterly earnings announcements (2022–2025) for simulated HOSE stocks: daily stock and market returns from day −150 to +10 around the announcement (day 0), the standardised unexpected earnings `sue`, and the `free_float` ratio. Returns are capped at ±7% (the HOSE price limit). Estimation window: days −150 to −31. Event window: (−1, +1).

In [ ]:
ev = pd.read_csv(DATA + "vn_earnings_events.csv")
# 1) Market model on the estimation window: ret = a + b * mkt_ret
est = ev[ev.day.between(-150, -31)]
coef = est.groupby("event_id")[["mkt_ret", "ret"]].apply(
    lambda g: pd.Series(np.polyfit(g["mkt_ret"], g["ret"], 1), index=["b", "a"]))
ev = ev.merge(coef, left_on="event_id", right_index=True)
# 2) Abnormal return = actual - normal return
ev["ar"] = ev["ret"] - (ev["a"] + ev["b"] * ev["mkt_ret"])
sigma = ev[ev.day.between(-150, -31)].groupby("event_id")["ar"].std(ddof=2).rename("sigma")
ev = ev.merge(sigma, left_on="event_id", right_index=True)
# 3) CAR over the event window (-1, +1), one row per event
car = ev[ev.day.between(-1, 1)].groupby("event_id").agg(car=("ar", "sum"), sue=("sue", "first"),
        free_float=("free_float", "first"), log_mcap=("log_mcap", "first"), sigma=("sigma", "first"))
car.describe().round(4)

In [ ]:
# Average abnormal returns by day, and the CAAR path for good- vs bad-news events
ev["news"] = np.where(ev["sue"] > 0, "Good news (SUE>0)", "Bad news (SUE<0)")
aar = ev[ev.day >= -10].groupby(["news", "day"])["ar"].mean().unstack(0)
aar.cumsum().plot(title="CAAR around earnings announcements"); plt.axvline(0, color="grey", ls="--"); plt.show()

### 2. Exercises

#### Exercise 10.1 – Is the CAAR significant?
Compute CAAR(−1,+1) separately for good-news and bad-news events. Test each against zero with (a) a cross-sectional t-test and (b) the Patell-type standardised test (divide each CAR by σ·√3).

In [ ]:
# YOUR CODE HERE


**Your interpretation:** _write 2–3 sentences here_

#### Exercise 10.2 – Cross-sectional regression
Regress CAR(−1,+1) on `sue`, `free_float`, their interaction and `log_mcap`, with robust SEs. Does the reaction to earnings news depend on free float? Interpret the interaction.

In [ ]:
# YOUR CODE HERE


**Your interpretation:** _write 2–3 sentences here_

#### Exercise 10.3 – Post-announcement drift and the price limit
Compute CAAR(+2,+10) for good- and bad-news events. Is there drift? Count how many daily returns in the event window hit ±7% and explain how the price limit could bias the measured reaction.

In [ ]:
# YOUR CODE HERE


**Your interpretation:** _write 2–3 sentences here_

---
**Before you leave:** File → Save a copy in Drive, then File → Download → .ipynb. Your notebook *is* your log: keep it with your A1/A2 working files.